<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/46_adaptive_tool_agent/adaptive_tool_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import defaultdict

In [ ]:
# Track tool success history
tool_performance = defaultdict(lambda: {"success": 0, "fail": 0})

In [ ]:
def classify_intent(query):
    query = query.lower()

    if any(word in query for word in ["%", "calculate", "sum", "add", "multiply"]):
        return "calculation"
    elif any(word in query for word in ["what is", "who is", "define"]):
        return "knowledge"
    else:
        return "explanation"

In [ ]:
def calculator_tool(query):
    match = re.search(r'(\d+)%.*?(\d+)', query)
    if match:
        percent = float(match.group(1))
        number = float(match.group(2))
        return (percent / 100) * number, True
    return "Calculation failed", False


def knowledge_tool(query):
    q = query.lower()

    if "artificial intelligence" in q:
        return "Artificial Intelligence is the simulation of human intelligence in machines.", True

    if "deep learning" in q:
        return "Deep Learning is a subset of machine learning that uses neural networks with multiple layers.", True

    return "Knowledge not found", False


def explanation_tool(query):
    if "machine learning" in query.lower():
        return "Machine Learning is a subset of AI that learns from data.", True
    return "Explanation not found", False

In [ ]:
def select_tool(intent):
    tool_map = {
    "calculation": ["calculator"],
    "knowledge": ["knowledge", "explanation"],
    "explanation": ["explanation", "knowledge"]  # important
}

    candidates = tool_map[intent]

    # choose best performing tool
    best_tool = max(
        candidates,
        key=lambda t: tool_performance[t]["success"] - tool_performance[t]["fail"]
    )

    return best_tool

In [ ]:
def update_performance(tool, success):
    if success:
        tool_performance[tool]["success"] += 1
    else:
        tool_performance[tool]["fail"] += 1

In [ ]:
def adaptive_agent(query):
    intent = classify_intent(query)
    tool = select_tool(intent)

    if tool == "calculator":
        answer, success = calculator_tool(query)
    elif tool == "knowledge":
        answer, success = knowledge_tool(query)
    else:
        answer, success = explanation_tool(query)

    update_performance(tool, success)

    return {
        "query": query,
        "intent": intent,
        "tool_used": tool,
        "answer": answer,
        "success": success,
        "tool_stats": dict(tool_performance)
    }

In [21]:
queries = [
    "What is 25% of 200?",              # success (calculator)
    "What is Artificial Intelligence?", # success (knowledge)
    "Explain machine learning",         # success (explanation)

    "Explain deep learning",            # explanation fails
    "What is deep learning?",           # knowledge succeeds → learning happens

    "Explain deep learning"             # NOW agent should prefer knowledge tool
]

for q in queries:
    result = adaptive_agent(q)
    print("\n======================")
    for k, v in result.items():
        print(f"{k.upper()}: {v}")


QUERY: What is 25% of 200?
INTENT: calculation
TOOL_USED: calculator
ANSWER: 50.0
SUCCESS: True
TOOL_STATS: {'calculator': {'success': 2, 'fail': 0}, 'knowledge': {'success': 3, 'fail': 1}, 'explanation': {'success': 0, 'fail': 1}}

QUERY: What is Artificial Intelligence?
INTENT: knowledge
TOOL_USED: knowledge
ANSWER: Artificial Intelligence is the simulation of human intelligence in machines.
SUCCESS: True
TOOL_STATS: {'calculator': {'success': 2, 'fail': 0}, 'knowledge': {'success': 4, 'fail': 1}, 'explanation': {'success': 0, 'fail': 1}}

QUERY: Explain machine learning
INTENT: explanation
TOOL_USED: knowledge
ANSWER: Knowledge not found
SUCCESS: False
TOOL_STATS: {'calculator': {'success': 2, 'fail': 0}, 'knowledge': {'success': 4, 'fail': 2}, 'explanation': {'success': 0, 'fail': 1}}

QUERY: Explain deep learning
INTENT: explanation
TOOL_USED: knowledge
ANSWER: Deep Learning is a subset of machine learning that uses neural networks with multiple layers.
SUCCESS: True
TOOL_STATS: 